In [1]:
!pip install crewai crewai_tools langchain_community sentence-transformers langchain-groq langchain_huggingface --quiet

In [2]:
!pip install langchain

In [1]:
import os
#os.environ['GROQ_API_KEY']='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'
os.environ['TAVILY_API_KEY']='tvly-z66T6o4dug8blrXv5eaZ4Ronsd3cVMZy___Q'


In [1]:
from langchain.agents import tool
from crewai_tools import PDFSearchTool
from crewai import Agent, Crew, Task,Process
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_openai import ChatOpenAI


/Users/tharhtet/.local/share/virtualenvs/Multi-Agent-Healthcare-iEESBots-Multimodal-MW7Qkosx/lib/python3.10/site-packages/pydantic/_internal/_config.py:295: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  warnings.warn(DEPRECATION_MESSAGE, DeprecationWarning)
/Users/tharhtet/.local/share/virtualenvs/Multi-Agent-Healthcare-iEESBots-Multimodal-MW7Qkosx/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:502: UserWarning: <built-in function callable> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(
/Users/tharhtet/.local/share/virtualenvs/Multi-Agent-Healthcare-iEESBots-Multimodal

In [2]:

"""
llm =ChatOpenAI(
    openai_api_key=os.environ['GROQ_API_KEY'],
    openai_api_base="https://api.groq.com/openai/v1",
    #model_name="llama3-8b-8192",
    model_name="llama-3.2-3b-preview",
    temperature=0.1,
    max_tokens=1000,

)
"""
# OR Use Ollama to run Locally
from crewai import Agent, Task, Process, Crew, LLM
ollama_llm = LLM(
            model="ollama/llama3.2:latest",
            api_base="http://localhost:11434"
            )


In [3]:
#!pip install tf-keras

In [4]:


rag_tool = PDFSearchTool(pdf='attentions_is_all_you_need.pdf',
    config=dict(
        llm=dict(
            provider="ollama", # or google, openai, anthropic, llama2, ...
            config=dict(
                model="llama3.2:latest",
                base_url="http://localhost:11434"
            ),
        ),
        embedder=dict(
                provider="huggingface", # or openai, ollama, ...
                config=dict(
                    model="BAAI/bge-small-en-v1.5",
                    #task_type="retrieval_document",
                    # title="Embeddings",
                ),
            ),
    )
)


/opt/anaconda3/envs/dev_env/lib/python3.10/abc.py:123: ResourceWarning: unclosed <socket.socket fd=77, family=AddressFamily.AF_INET, type=SocketKind.SOCK_STREAM, proto=6, laddr=('127.0.0.1', 62070), raddr=('127.0.0.1', 11434)>
  return _abc_subclasscheck(cls, subclass)
/Users/tharhtet/.local/share/virtualenvs/Multi-Agent-Healthcare-iEESBots-Multimodal-MW7Qkosx/lib/python3.10/site-packages/embedchain/embedder/huggingface.py:34: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=self.config.model, model_kwargs=self.config.model_kwargs)
/Users/tharhtet/.local/share/virtualenvs/Multi-Agent-Healthcare-iEESBots-Multimo

In [5]:
question = "How did self-attention mechanism evolve in large language models in pdf?"
rag_tool.run(question)

Using Tool: Search a PDF's content


'Relevant Content:\nmakes it more difﬁcult to learn dependencies between distant positions [ 11]. In the Transformer this is reduced to a constant number of operations, albeit at the cost of reduced effective resolution due to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as described in section 3.2. Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. Self-attention has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent sentence representations [4, 22, 23, 19]. End-to-end memory networks are based on a recurrent attention mechanism instead of sequence- aligned recurrence and have been shown to perform well on simple-language question answering and language modeling tasks [28]. To the best of our knowledge, however, t

In [6]:
# messages = [{"role": "user", "content": "hello"}]  # Format message for chat
# response = ollama_llm.call("what is tensorflow") # Get response and clean it
# response

In [7]:


@tool
def main_router_tool(question : str) -> str:
  """Routes queries to the appropriate resources"""


  print("prompt : ",question)

  sample_data = """
  Car Parking-1 is still avaialbe and 4 parking spaces are free.
  Emergency call number is 062-659-7737
  Hostpital address is Bangkok, Thailand
  """



  prompt = f"""
    Your are a router. Your task is classify the following question into one of the four categories:
    
    1. Text-to-Image (Generating an image from text)
    2. Text-to-Speech (Converting text into speech)
    3. vectorstore (If this question's concept is related with this phragraph  {sample_data})
    4. Web Search (General question requiring online search)
    Question: {question}
    

    ### Response Instructions:
    - Reply with ONLY a single digit: 1, 2, 3, or 4.
    - Do NOT include explanations, preambles, or extra words.
    - Output should be exactly one character (e.g., "1", "2", "3", or "4").

    ### Example Outputs:
      must be only one digit
      1  
      2
      3
      4
    """
  """
  messages = [
    {
      "role": "system",
      "content": prompt
    },
    {
      "role": "user",
      "content": question
    }
  ]
  """

  print("message : ",prompt)

  
  response = ollama_llm.call(prompt)
 
  if response == "1":
    return 'text2image'
  elif response == "2":
    return 'text2speech'
  elif response == "3":
    return 'vectorstore'
  elif response == "4":
    return 'web_search'





In [8]:
image_prompt = "How look like BMW car?"
task_route = main_router_tool(image_prompt)
print(task_route)

image_prompt = "What are the difference between tensorflow and pytorch?"
task_route = main_router_tool(image_prompt)
print(task_route)


image_prompt = "can you describe a house with beautiful garden?"
task_route = main_router_tool(image_prompt)
print(task_route)


prompt = "Is still parking available?"
task_route = main_router_tool(prompt)
print(task_route)



prompt = "What is emergency number?"
task_route = main_router_tool(prompt)
print(task_route)

/var/folders/xd/3z5vvpds0zxf_pypxd4cn3d80000gn/T/ipykernel_45788/3551826645.py:2: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use :meth:`~invoke` instead.
  task_route = main_router_tool(image_prompt)


prompt :  How look like BMW car?
message :  
    Your are a router. Your task is classify the following question into one of the four categories:
    
    1. Text-to-Image (Generating an image from text)
    2. Text-to-Speech (Converting text into speech)
    3. vectorstore (If this question's concept is related with this phragraph  
  Car Parking-1 is still avaialbe and 4 parking spaces are free.
  Emergency call number is 062-659-7737
  Hostpital address is Bangkok, Thailand
  )
    4. Web Search (General question requiring online search)
    Question: How look like BMW car?
    

    ### Response Instructions:
    - Reply with ONLY a single digit: 1, 2, 3, or 4.
    - Do NOT include explanations, preambles, or extra words.
    - Output should be exactly one character (e.g., "1", "2", "3", or "4").

    ### Example Outputs:
      must be only one digit
      1  
      2
      3
      4
    
text2image
prompt :  What are the difference between tensorflow and pytorch?
message :  
    Y

In [9]:
import yaml

def config_reader(file_path):
    with open(file_path, 'r') as f:
        data = yaml.load(f, Loader=yaml.SafeLoader)
        
    # Print the values as a dictionary
    return data


In [10]:
from crewai.project import CrewBase, agent, crew, task

In [14]:


# Router Agent
agent1_config = "config/agents.yaml"

router_agent = Agent(
                    config=config_reader(agent1_config)["router_agent"],
                    allow_delegation=False,
                    verbose=True,
                    llm=ollama_llm,
                    
                    )
    







In [15]:
#Task 1
router_task = Task(
    description=("Analyze the given question and determine its category"
    ),
    expected_output=("A string representing the correct task category (text2image, image2text, text2speech, web_search)."),
    agent=router_agent,
    tools = [main_router_tool]
    
)

ValidationError: 1 validation error for Task
tools.0
  Input should be a valid dictionary or instance of BaseTool [type=model_type, input_value=StructuredTool(name='main...er_tool at 0x34cfbb6d0>), input_type=StructuredTool]
    For further information visit https://errors.pydantic.dev/2.10/v/model_type

In [16]:
crew1 = Crew(
    agents = [router_agent],
    tasks = [router_task],
    verbose= True,
)

test_input =  {'question': {'description': " Is still parking available?", 'type': 'str'}}

#test_input =  {'question': {'description': "Create an image of a futuristic cityscape at sunset.", 'type': 'str'}}

test_input = {'question': {'description': "Is still parking available?", 'type': 'str'}}

result1 = crew1.kickoff(inputs=test_input)

2025-02-25 17:11:28,059 - 8428914752 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


# Agent: Router Agent
## Task: Analyze the given question and determine its category
 

I encountered an error while trying to use the tool. This was the error: Arguments validation failed: 1 validation error for main_router_tool
question
  Input should be a valid string [type=string_type, input_value={'description': 'None', 'type': 'str'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/string_type.
 Tool main_router_tool accepts these inputs: Tool Name: main_router_tool
Tool Arguments: {'question': {'description': None, 'type': 'str'}}
Tool Description: Routes queries to the appropriate resources



# Agent: Router Agent
## Using tool: main_router_tool
## Tool Input: 
"{\"question\": {\"description\": \"None\", \"type\": \"str\"}}"
## Tool Output: 

I encountered an error while trying to use the tool. This was the error: Arguments validation failed: 1 validation error for main_router_tool
question
  Input should be a valid string [type=string_typ

KeyboardInterrupt: 

In [17]:
# Define processing agents
text2image_agent = Agent(
    role="Image Generator",
    goal="Generate an image based on a text prompt.",
    backstory="A skilled AI artist capable of bringing textual descriptions to life.",
    llm=ollama_llm
)

image2text_agent = Agent(
    role="Image Analyst",
    goal="Extract meaningful text or descriptions from images.",
    backstory="An AI expert in image recognition and analysis.",
    llm=ollama_llm
)

text2speech_agent = Agent(
    role="Speech Synthesizer",
    goal="Convert text into natural-sounding speech.",
    backstory="A digital voice assistant specializing in realistic text-to-speech conversion.",
    llm=ollama_llm
)

web_search_agent = Agent(
    role="Web Researcher",
    goal="Search the web for the most relevant and up-to-date information.",
    backstory="An AI-powered researcher who retrieves information quickly and accurately.",
    llm=ollama_llm
)


In [27]:
# Task for the router agent to decide and delegate
route_task_task = Task(
    description="Analyze the user's question and determine which agent should handle it.",
    agent=router_agent,
    expected_output="A string representing the correct task category (text2image, image2text, text2speech, web_search).",
    tools= [main_router_tool]
)

crew = Crew(agents=[router_agent, text2image_agent, image2text_agent, text2speech_agent, web_search_agent],
            tasks=[route_task_task])

ValidationError: 1 validation error for Task
tools.0
  Input should be a valid dictionary or instance of BaseTool [type=model_type, input_value=StructuredTool(name='main...er_tool at 0x34cfbb6d0>), input_type=StructuredTool]
    For further information visit https://errors.pydantic.dev/2.10/v/model_type

In [26]:
# Example user query
user_query = "generate a image"

# Execute routing decision
task_result = crew.kickoff(inputs={"question": user_query})  # Router determines the task


# Agent: Router Agent
## Task: Analyze the user's question and determine which agent should handle it.


# Agent: Router Agent
## Final Answer: 
Here is the response from the text2speech agent:

"This conversation is critical to me, as it affects my performance and overall effectiveness in providing accurate and helpful responses to users. I must use the available tools and resources to generate a comprehensive and informative answer that meets the user's needs."


